# 🖐️ Fingerprint Processing Workshop

---
## 1. Load Photo & Select Area
Put your finger photo in `data/`, pick it from the dropdown, drag to select the fingerprint area, then click **Confirm ROI**.

## Background

This notebook is a hands-on reproduction of a real-world attack demonstrated by Jan Krißler (alias *Starbug*) of the Chaos Computer Club in 2014, where a politician's fingerprint was reconstructed from ordinary press-conference photographs — without ever touching her finger.

**Further reading:**

- BBC News — [Hacker 'fakes' German minister's fingerprints using photos](https://www.bbc.com/news/technology-30623611)
- Ars Technica — [Politician's fingerprint reproduced using photos of her hands](https://arstechnica.com/information-technology/2014/12/politicians-fingerprint-reproduced-using-photos-of-her-hands/)

---

In [1]:
import sys, importlib, inspect, re
from pathlib import Path
from typing import Any, Dict, Tuple

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.path import Path as MplPath
from matplotlib.widgets import PolygonSelector
from IPython.display import display as _display, HTML as _HTML

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in ('notebooks', 'notebooks_final') else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.interactive as ip
import src.utils as fputils

try:
    import src.enhancement as enh
except Exception:
    enh = None
try:
    import src.mask as fpmask
except Exception:
    fpmask = None
try:
    import src.binarization as fpbin
except Exception:
    fpbin = None
try:
    import src.morphology as fpmorph
except Exception:
    fpmorph = None

for _mod in [ip, fputils, enh, fpmask, fpbin, fpmorph]:
    if _mod is not None:
        importlib.reload(_mod)

ensure_odd     = fputils.ensure_odd
normalize_gray = fputils.normalize_gray

# Remove Jupyter's default output-cell scroll limit so the pipeline renders at full height
_display(_HTML("""<style>
.jp-OutputArea-output { max-height: none !important; overflow: visible !important; }
.output_scroll { height: unset !important; }
div.output_area { overflow: visible !important; }
</style>"""))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('All modules loaded.')

PROJECT_ROOT: /home/x232886/PycharmProjects/poznejfi
All modules loaded.


In [2]:
import re as _re, io as _io, time as _time
import ipywidgets as _w, numpy as np
from PIL import Image as _PilImage
from IPython.display import display as _display

PHOTOS_DIR           = PROJECT_ROOT / 'data'
PHOTOS_DIR.mkdir(parents=True, exist_ok=True)
SCREEN_PX_PER_MM     = 1 / 0.2745
PRINTER_DPI          = 600.0
FINGER_WIDTH_MM_INIT = 18.0
GRID_MM_W, GRID_MM_H = 60, 55
_IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}

IMAGE_PATH = img = polygon_mask = None

class _CropState:
    _dpi = 0.0
_crop_widget = _CropState()


def _make_roi_widget(image, path):
    from ipycanvas import MultiCanvas, hold_canvas as _hc

    h, w  = image.shape[:2]
    scale = min(150 / w, 200 / h)
    cw, ch = max(1, int(w * scale)), max(1, int(h * scale))

    img8 = normalize_gray(image)
    pil  = _PilImage.fromarray(img8).resize((cw, ch), _PilImage.LANCZOS)
    buf  = _io.BytesIO(); pil.save(buf, 'PNG')
    bg   = _w.Image(value=buf.getvalue(), format='png',
                    layout=_w.Layout(width=f'{cw}px', height=f'{ch}px'))

    canvas = MultiCanvas(2, width=cw, height=ch)
    canvas.layout.width  = f'{cw}px'
    canvas.layout.height = f'{ch}px'
    canvas.layout.border = '1px solid #aaa'
    canvas.layout.cursor = 'crosshair'

    st      = {'p1': None, 'p2': None, 'drag': False}
    status  = _w.Label('Drag to select the fingerprint area.')
    result  = _w.HTML('')
    preview = _w.Output()
    _last   = [0.0]

    def _draw(ex=None, ey=None):
        if not st['p1']: return
        x2, y2 = (ex, ey) if ex is not None else st['p2']
        x0, y0 = min(st['p1'][0], x2), min(st['p1'][1], y2)
        x1, y1 = max(st['p1'][0], x2), max(st['p1'][1], y2)
        with _hc(canvas[1]):
            canvas[1].clear()
            canvas[1].fill_style = 'rgba(0,200,255,0.15)'
            canvas[1].fill_rect(x0, y0, x1-x0, y1-y0)
            canvas[1].stroke_style = 'cyan'; canvas[1].line_width = 1
            canvas[1].stroke_rect(x0, y0, x1-x0, y1-y0)

    def _down(x, y): st['p1'] = st['p2'] = (x, y); st['drag'] = True
    def _move(x, y):
        now = _time.monotonic()
        if now - _last[0] < 0.05: return
        _last[0] = now
        if st['drag']: st['p2'] = (x, y); _draw(x, y)
    def _up(x, y):
        st['p2'] = (x, y); st['drag'] = False; _draw()
        status.value = 'Done — click Confirm ROI.'

    canvas.on_mouse_down(_down); canvas.on_mouse_move(_move); canvas.on_mouse_up(_up)

    btn = _w.Button(description='Confirm ROI', button_style='primary',
                    layout=_w.Layout(width='120px'))

    def _confirm(_):
        global polygon_mask, img
        if not st['p1'] or not st['p2']: return
        s = scale
        ix0 = max(0, int(min(st['p1'][0], st['p2'][0]) / s))
        iy0 = max(0, int(min(st['p1'][1], st['p2'][1]) / s))
        ix1 = min(w,  int(max(st['p1'][0], st['p2'][0]) / s))
        iy1 = min(h,  int(max(st['p1'][1], st['p2'][1]) / s))

        crop = image[iy0:iy1, ix0:ix1]
        out_path = path.parent / f'{path.stem}_processed.png'
        cv2.imwrite(str(out_path), crop)

        img = crop
        mask = np.ones(crop.shape[:2], dtype=np.uint8) * 255
        polygon_mask = mask; globals()['polygon_mask'] = mask; globals()['img'] = crop
        try: pipeline.apply_to_image(crop, mask)
        except NameError: pass

        result.value = f"<span style='color:#28a745;font-weight:bold'>✓ Saved {out_path.name}</span>"
        preview.clear_output(wait=True)
        with preview:
            _display(_PilImage.fromarray(normalize_gray(crop)))

    btn.on_click(_confirm)
    widget = _w.VBox([status, canvas,
                      _w.HBox([btn, result], layout=_w.Layout(margin='4px 0 0 0')),
                      preview])
    return widget, canvas, bg, cw, ch


# ── File selector ─────────────────────────────────────────────────────────────
_all = sorted(p for p in PHOTOS_DIR.iterdir() if p.suffix.lower() in _IMAGE_EXTS)

if not _all:
    _display(_w.HTML(f"<span style='color:gray'>No images found in <code>{PHOTOS_DIR}</code></span>"))
else:
    _selector = _w.Dropdown(options=[(p.name, p) for p in _all], description='Photo:',
                            layout=_w.Layout(width='400px'))
    _load_btn = _w.Button(description='Load', button_style='primary')
    _out      = _w.Output()

    def _on_load(_):
        global IMAGE_PATH, img, polygon_mask
        p    = _selector.value
        _img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if _img is None: return
        try:
            _di = _PilImage.open(p).info.get('dpi', (0, 0))
            _crop_widget._dpi = float(_di[0]) if isinstance(_di, (tuple, list)) else float(_di)
        except Exception:
            _crop_widget._dpi = 0.0
        IMAGE_PATH = p; img = _img
        polygon_mask = np.ones(_img.shape[:2], dtype=np.uint8) * 255
        _out.clear_output(wait=True)
        with _out:
            widget, canvas, bg, cw, ch = _make_roi_widget(_img, p)
            _display(widget)
            canvas[0].draw_image(bg, 0, 0, cw, ch)

    _load_btn.on_click(_on_load)
    _display(_w.VBox([_selector, _load_btn, _out]))

---
## 2. Process Ridges
Adjust the sliders until the ridge/valley map looks clean.

In [3]:
def call_kwargs(func, /, **kwargs):
    """Call func with only the kwargs it accepts."""
    allowed = set(inspect.signature(func).parameters)
    return func(**{k: v for k, v in kwargs.items() if k in allowed})


def widget_from_spec(spec: Dict[str, Any]):
    kind = spec['kind']
    if kind == 'int':
        return widgets.IntSlider(
            value=spec['value'], min=spec['min'], max=spec['max'],
            step=spec.get('step', 1), description=spec['label'], continuous_update=False
        )
    if kind == 'float':
        return widgets.FloatSlider(
            value=spec['value'], min=spec['min'], max=spec['max'],
            step=spec.get('step', 0.1), description=spec['label'], continuous_update=False
        )
    if kind == 'bool':
        return widgets.Checkbox(value=spec['value'], description=spec['label'], indent=False)
    if kind == 'choice':
        return widgets.Dropdown(options=spec['options'], value=spec['value'], description=spec['label'])
    raise ValueError(f'Unknown kind: {kind}')


def wspec(specs):
    return {k: widget_from_spec(v) for k, v in specs.items()}


def step_crop_to_mask(image, polygon_mask=None):
    if polygon_mask is None or not (polygon_mask > 0).any():
        return image, polygon_mask
    rows = np.any(polygon_mask > 0, axis=1)
    cols = np.any(polygon_mask > 0, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    img_crop  = image[rmin:rmax + 1, cmin:cmax + 1].copy()
    mask_crop = polygon_mask[rmin:rmax + 1, cmin:cmax + 1].copy()
    img_crop[mask_crop == 0] = 0
    return img_crop, mask_crop


def step_clahe(image, clip=2.0, tile=8):
    img8 = normalize_gray(image)
    if enh is not None and hasattr(enh, 'apply_clahe'):
        return call_kwargs(enh.apply_clahe, img_gray=img8, clip=clip, tile=tile)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
    return clahe.apply(img8)


def step_median(image, ksize=0):
    if ksize <= 0:
        return image
    k = ensure_odd(ksize, 1)
    return cv2.medianBlur(normalize_gray(image), k)


_BIN_ACTIVE_PARAMS = {
    'adaptive_gaussian_blurred': {'block', 'C', 'blur'},
    'adaptive_gaussian':         {'block', 'C'},
    'adaptive':                  {'block', 'C'},
    'adaptive_mean':             {'block', 'C'},
    'sauvola':                   {'block'},
    'otsu':                      set(),
    'fixed':                     {'thresh'},
}

def step_binarize(image, method='adaptive_gaussian_blurred', block=21, C=2, blur=3, thresh=127):
    img8 = normalize_gray(image)
    if fpbin is not None:
        fn_map = {
            'fixed':                     'fixed_binarize',
            'otsu':                      'otsu_binarize',
            'adaptive':                  'adaptive_binarize',
            'adaptive_mean':             'adaptive_mean_binarize',
            'adaptive_gaussian':         'adaptive_gaussian_binarize',
            'adaptive_gaussian_blurred': 'adaptive_gaussian_binarize_blurred',
            'sauvola':                   'sauvola_binarize',
        }
        fn_name = fn_map.get(method)
        fn = getattr(fpbin, fn_name, None) if fn_name else None
        if fn is not None:
            return call_kwargs(
                fn, img_gray=img8, gray=img8, img=img8,
                thresh=thresh, block_size=block, window_size=block,
                C=C, blur_ksize=blur, invert=True
            )
    if method == 'fixed':
        _, out = cv2.threshold(img8, thresh, 255, cv2.THRESH_BINARY)
        return out
    if method == 'otsu':
        _, out = cv2.threshold(img8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return out
    b = ensure_odd(block, 3)
    src = cv2.GaussianBlur(img8, (ensure_odd(blur, 1),) * 2, 0)
    return cv2.adaptiveThreshold(src, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, b, C)


def step_morphology(binary, polygon_mask=None, close_k=3, open_k=3, fill_holes=True, remove_small=0):
    out = (binary > 0).astype(np.uint8) * 255
    if close_k > 0:
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ensure_odd(close_k, 1),) * 2)
        out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, ker)
    if open_k > 0:
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ensure_odd(open_k, 1),) * 2)
        out = cv2.morphologyEx(out, cv2.MORPH_OPEN, ker)
    if fill_holes:
        h, w = out.shape
        flood = (255 - out).copy()
        mask_ff = np.zeros((h + 2, w + 2), np.uint8)
        cv2.floodFill(flood, mask_ff, (0, 0), 128)
        holes = (flood == 255).astype(np.uint8)
        out = np.maximum(out, holes * 255).astype(np.uint8)
    if remove_small > 0:
        n, labels, stats, _ = cv2.connectedComponentsWithStats((out > 0).astype(np.uint8), 8)
        clean = np.zeros_like(out)
        for i in range(1, n):
            if stats[i, cv2.CC_STAT_AREA] >= remove_small:
                clean[labels == i] = 255
        out = clean
    if polygon_mask is not None:
        out[polygon_mask == 0] = 0
    return out


def step_ridge_valley(binary, polygon_mask=None, src_image=None):
    mask   = (polygon_mask > 0) if polygon_mask is not None else np.ones(binary.shape, bool)
    ridge  = ((binary > 0) & mask).astype(np.uint8) * 255
    valley = ((binary == 0) & mask).astype(np.uint8) * 255
    h, w   = binary.shape
    vis    = np.full((h, w, 3), 200, dtype=np.uint8)
    vis[mask] = 255
    vis[(ridge > 0)]  = (220, 60, 60)
    vis[(valley > 0)] = (60, 100, 220)
    valley_print = np.full((h, w), 255, dtype=np.uint8)
    valley_print[(binary == 0) & mask] = 0
    src8 = normalize_gray(src_image) if src_image is not None else np.full((h, w), 200, dtype=np.uint8)
    valley_overlay = cv2.cvtColor(src8, cv2.COLOR_GRAY2BGR)
    valley_overlay[(binary == 0) & mask] = (255, 0, 0)
    return ridge, valley, vis, valley_print, valley_overlay

def _step_ridge_valley_wrapper(binary, polygon_mask=None, src_image=None):
    return step_ridge_valley(binary, polygon_mask, src_image)

def step_display_overlay(valley_overlay):
    return valley_overlay


# ── Step 2+3: Pipeline ────────────────────────────────────────────────────────

class RidgeValleyPipeline(ip.InteractivePipeline):
    def __init__(self, *args, polygon_mask: np.ndarray = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.polygon_mask = polygon_mask

    def run_from(self, start_index: int = 0):
        if self.initial_image is None:
            return
        self.context.reset(self.initial_image)
        self.context['polygon_mask'] = self.polygon_mask
        for i, step in enumerate(self.steps):
            ok = step.run(self.context)
            if not ok:
                for j in range(i + 1, len(self.steps)):
                    self.steps[j].status_label.value = 'Blocked by previous error'
                break

    def run(self):
        self.run_from(0)

    def update_mask(self, new_mask: np.ndarray):
        self.polygon_mask = new_mask
        self.run()

    def apply_to_image(self, new_image: np.ndarray, new_mask: np.ndarray = None):
        self.initial_image = new_image
        if new_mask is not None:
            self.polygon_mask = new_mask
        self.run()


_COL_ON  = "#4a90d9"
_COL_OFF = "#bdbdbd"

def _bar_html(color):
    return (f"<div style='width:4px;min-width:4px;height:22px;"
            f"background:{color};border-radius:2px;margin-right:4px'></div>")

def _param_bar(color):
    return widgets.HTML(_bar_html(color))

def _param_row(bar_widget, w):
    return widgets.HBox([bar_widget, w], layout=widgets.Layout(align_items='center'))


_STEPS = [
    {
        'name': '0. Crop',
        'func': step_crop_to_mask,
        'input_map': {'image': 'image', 'polygon_mask': 'polygon_mask'},
        'output_keys': ['image', 'polygon_mask'],
        'display_key': 'image',
        'enabled': True,
        'params': {},
    },
    {
        'name': '1. Contrast',
        'func': step_clahe,
        'input_map': {'image': 'image'},
        'output_keys': ['image'],
        'enabled': True,
        'params': wspec({
            'clip': {'kind': 'float', 'value': 10.0, 'min': 0.5, 'max': 40.0, 'step': 0.5, 'label': 'Clip'},
            'tile': {'kind': 'int',   'value': 8,    'min': 2,   'max': 32,   'step': 1,   'label': 'Tile'},
        }),
    },
    {
        'name': '2. Smooth',
        'func': step_median,
        'input_map': {'image': 'image'},
        'output_keys': ['image'],
        'enabled': False,
        'params': wspec({
            'ksize': {'kind': 'int', 'value': 0, 'min': 0, 'max': 14, 'step': 2, 'label': 'K (0=off)'},
        }),
    },
    {
        'name': '3. Binarize',
        'func': step_binarize,
        'input_map': {'image': 'image'},
        'output_keys': ['binary'],
        'display_key': 'binary',
        'enabled': True,
        'params': wspec({
            'method': {'kind': 'choice', 'value': 'sauvola',
                       'options': ['adaptive_gaussian_blurred', 'adaptive_gaussian', 'adaptive',
                                   'adaptive_mean', 'sauvola', 'otsu', 'fixed'],
                       'label': 'Method'},
            'block':  {'kind': 'int',   'value': 35,  'min': 3,   'max': 101, 'step': 2,   'label': 'Block'},
            'C':      {'kind': 'int',   'value': 2,   'min': -20, 'max': 20,  'step': 1,   'label': 'C'},
            'blur':   {'kind': 'int',   'value': 3,   'min': 1,   'max': 21,  'step': 2,   'label': 'Blur'},
            'thresh': {'kind': 'int',   'value': 127, 'min': 0,   'max': 255, 'step': 1,   'label': 'Thresh'},
        }),
        'param_visibility': {'method': _BIN_ACTIVE_PARAMS},
    },
    {
        'name': '4. Cleanup',
        'func': step_morphology,
        'input_map': {'binary': 'binary', 'polygon_mask': 'polygon_mask'},
        'output_keys': ['binary'],
        'display_key': 'binary',
        'enabled': True,
        'params': wspec({
            'close_k':      {'kind': 'int',  'value': 2,    'min': 0, 'max': 21,   'step': 2, 'label': 'Close'},
            'open_k':       {'kind': 'int',  'value': 4,    'min': 0, 'max': 21,   'step': 2, 'label': 'Open'},
            'fill_holes':   {'kind': 'bool', 'value': False,                                    'label': 'Fill holes'},
            'remove_small': {'kind': 'int',  'value': 0,    'min': 0, 'max': 2000, 'step': 10,'label': 'Min area'},
        }),
    },
    {
        'name': '5. Valley map',
        'func': _step_ridge_valley_wrapper,
        'input_map': {'binary': 'binary', 'polygon_mask': 'polygon_mask', 'src_image': 'image'},
        'output_keys': ['ridge_map', 'valley_map', 'rv_vis', 'valley_print', 'valley_overlay'],
        'display_key': 'valley_print',
        'enabled': True,
        'params': {},
    },
    {
        'name': '6. Overlay',
        'func': step_display_overlay,
        'input_map': {'valley_overlay': 'valley_overlay'},
        'output_keys': ['valley_overlay'],
        'display_key': 'valley_overlay',
        'enabled': True,
        'params': {},
    },
]


def _make_step(spec):
    input_map        = spec.get('input_map', {})
    display_key      = spec.get('display_key')
    param_visibility = spec.get('param_visibility', {})

    class MappedStep(ip.Step):
        def get_ui(self, layout_mode: str = 'fixed', image_scale: float = 1.0, **kwargs):
            size = int(300 * image_scale)
            self.image_widget.width  = f'{size}px'
            self.image_widget.height = f'{size}px'
            if not self.enabled and layout_mode == 'dynamic':
                return None
            _bars = {}
            param_rows = [self.checkbox]
            for pname, w in self.params.items():
                if isinstance(w, widgets.Widget):
                    bar = _param_bar(_COL_ON)
                    _bars[pname] = bar
                    param_rows.append(_param_row(bar, w))
            for selector_name, visibility_map in param_visibility.items():
                selector_w = self.params.get(selector_name)
                if not isinstance(selector_w, widgets.Widget):
                    continue
                def _refresh_bars(change, _vm=visibility_map, _b=_bars, _sn=selector_name):
                    active = _vm.get(change['new'], set())
                    for pn, bar in _b.items():
                        if pn == _sn:
                            continue
                        bar.value = _bar_html(_COL_ON if pn in active else _COL_OFF)
                selector_w.observe(_refresh_bars, names='value')
                active = visibility_map.get(selector_w.value, set())
                for pn, bar in _bars.items():
                    if pn != selector_name:
                        bar.value = _bar_html(_COL_ON if pn in active else _COL_OFF)
            self.controls_container = widgets.VBox(
                param_rows, layout=widgets.Layout(width='100%'))
            if not self.enabled and layout_mode == 'fixed':
                self.controls_container.layout.display = 'none'
                placeholder = widgets.HTML(
                    f"<div style='width:{size}px;height:{size}px;background:#eee;"
                    f"display:flex;align-items:center;justify-content:center;"
                    f"color:#aaa'>Skipped<br>({self.name})</div>")
                return widgets.VBox(
                    [widgets.Label(self.name), placeholder, self.checkbox],
                    layout=widgets.Layout(border='1px solid #ddd', margin='5px', padding='5px'))
            self.controls_container.layout.display = 'flex'
            return widgets.VBox(
                [widgets.Label(self.name, style={'font_weight': 'bold'}),
                 self.image_widget,
                 self.controls_container,
                 self.status_label],
                layout=widgets.Layout(
                    border='1px solid #888', margin='5px', padding='5px',
                    width=f'{size + 20}px'))

        def run(self, context):
            if not self.enabled:
                self.status_label.value = 'Skipped'
                return True
            kwargs = {}
            for arg_name, ctx_key in input_map.items():
                val = context.get(ctx_key)
                if val is None and 'mask' not in arg_name:
                    self.status_label.value = f'Missing ctx key: {ctx_key}'
                    return False
                kwargs[arg_name] = val
            for pname, wobj in self.params.items():
                import ipywidgets as _w
                kwargs[pname] = wobj.value if isinstance(wobj, _w.Widget) else wobj
            try:
                import time as _t
                t0 = _t.time()
                result = self.func(**kwargs)
                dt = _t.time() - t0
                self.status_label.value = f'{dt*1000:.0f} ms'
                if len(self.output_keys) == 1:
                    context[self.output_keys[0]] = result
                    primary = result
                else:
                    if not isinstance(result, tuple):
                        context[self.output_keys[0]] = result
                        primary = result
                    else:
                        for i, k in enumerate(self.output_keys):
                            if i < len(result):
                                context[k] = result[i]
                        primary = result[0]
                disp = context.get(display_key) if display_key else primary
                if isinstance(disp, np.ndarray):
                    self.image_widget.value = ip._array_to_bytes(disp)
                return True
            except Exception as exc:
                self.status_label.value = f'Error: {exc}'
                return False

    return MappedStep(
        name=spec['name'],
        func=spec['func'],
        input_keys=list(spec.get('input_map', {}).values()),
        output_keys=spec['output_keys'],
        params=spec.get('params', {}),
        enabled=spec.get('enabled', True),
    )


steps    = [_make_step(s) for s in _STEPS]
pipeline = RidgeValleyPipeline(
    steps=steps,
    initial_image=img,
    polygon_mask=polygon_mask,
    screen_px_per_mm=SCREEN_PX_PER_MM,
)
display(pipeline.display())

---
## 3. Real Size & Save
Select your monitor, hold your finger against the screen, drag **Width** to match, then save.

In [4]:
class ScaleCalibrationWidget:
    _RESCALE_KEYS = ["binary", "ridge_map", "valley_map", "rv_vis", "valley_print"]
    _SCREEN_PX_MM = SCREEN_PX_PER_MM
    _GRID_MM_W  = GRID_MM_W
    _GRID_MM_H  = GRID_MM_H

    def __init__(self, context):
        from ipycanvas import MultiCanvas, hold_canvas as _hc
        from PIL import Image as PilImage
        import io as _io

        self._PilImage  = PilImage
        self._io        = _io
        self._hc        = _hc
        self.context    = context
        self._scale_factor = None

        px_mm = self._SCREEN_PX_MM
        self._px_mm = px_mm
        self._gw = int(self._GRID_MM_W * px_mm)

        self._orig_arrays = {}
        for key in self._RESCALE_KEYS:
            arr = context.get(key)
            if isinstance(arr, np.ndarray):
                self._orig_arrays[key] = arr.copy()

        img = next((self._orig_arrays.get(k) for k in ("valley_map", "binary", "image")
                    if self._orig_arrays.get(k) is not None), None)
        self._img = img

        if img is not None:
            _ih, _iw = img.shape[:2]
            _max_h_mm = 60.0 * _ih / _iw
            self._gh = max(int(self._GRID_MM_H * px_mm), int(_max_h_mm * px_mm))
        else:
            self._gh = int(self._GRID_MM_H * px_mm)

        self._mesh_canvas = MultiCanvas(2, width=self._gw, height=self._gh)
        self._mesh_canvas.layout.border = "1px solid #888"

        self._slider = widgets.FloatSlider(
            value=FINGER_WIDTH_MM_INIT, min=5.0, max=60.0, step=0.5,
            description="Width (mm):",
            continuous_update=False,
            layout=widgets.Layout(width="320px"),
            style={"description_width": "90px"})
        self._slider.observe(self._on_slider, names="value")

        _cw = globals().get('_crop_widget')
        _img_dpi = getattr(_cw, '_dpi', 0.0) if _cw is not None else 0.0
        _img_dpi = _img_dpi if _img_dpi >= 150 else 0.0
        self._img_dpi = _img_dpi
        if _img_dpi > 0 and img is not None:
            _known_w_mm = img.shape[1] * 25.4 / _img_dpi
            _known_h_mm = img.shape[0] * 25.4 / _img_dpi
            if 5.0 <= _known_w_mm <= 55.0:
                self._slider.value = _known_w_mm
                _img_dpi_html = (f"Input image: <b>{_img_dpi:.0f} DPI</b>"
                                 f" &nbsp;→&nbsp; <b>{_known_w_mm:.1f} × {_known_h_mm:.1f} mm</b>"
                                 f" &nbsp;({img.shape[1]} × {img.shape[0]} px)"
                                 f" — slider pre-set")
            else:
                _img_dpi_html = (f"Input image: <b>{_img_dpi:.0f} DPI</b>"
                                 f" &nbsp;→&nbsp; computed size {_known_w_mm:.0f} mm"
                                 f" — out of range, adjust slider manually")
        else:
            _img_dpi_html = "Input image: <i>DPI not embedded — adjust slider manually</i>"
        self._img_dpi_label = widgets.HTML(
            f"<small style='color:#555'>{_img_dpi_html}</small>")

        self._size_result = widgets.HTML(value="")

        _controls = widgets.VBox([
            widgets.HTML("<b>Hold your finger on the screen.<br>"
                         "Drag Width to match — size is applied automatically.</b>"),
            widgets.HTML("<hr style='margin:6px 0'>"),
            self._slider,
            self._img_dpi_label,
            self._size_result,
        ], layout=widgets.Layout(padding="8px", min_width="340px"))

        self.inner = widgets.HBox(
            [self._mesh_canvas, _controls],
            layout=widgets.Layout(align_items="flex-start", gap="16px"))

    def _dpi_value(self):
        return PRINTER_DPI

    def _make_grid_pil(self):
        px = self._px_mm
        gw, gh = self._gw, self._gh
        img = self._PilImage.new("RGB", (gw, gh), "white")
        from PIL import ImageDraw
        draw = ImageDraw.Draw(img)
        for mm in range(0, self._GRID_MM_W + 1):
            x = int(mm * px)
            major = mm % 5 == 0
            color = "#888" if major else "#e0e0e0"
            draw.line([(x, 0), (x, gh)], fill=color, width=1)
            if major and mm > 0:
                draw.text((x + 2, 2), f"{mm}", fill="#666")
        for mm in range(0, self._GRID_MM_H + 1):
            y = int(mm * px)
            major = mm % 5 == 0
            color = "#888" if major else "#e0e0e0"
            draw.line([(0, y), (gw, y)], fill=color, width=1)
            if major and mm > 0:
                draw.text((2, y + 2), f"{mm}", fill="#666")
        return img

    def _draw_grid(self):
        import io as _io2
        grid_pil = self._make_grid_pil()
        buf = _io2.BytesIO(); grid_pil.save(buf, format="PNG")
        iw = widgets.Image(value=buf.getvalue(), format="png")
        self._mesh_canvas[0].draw_image(iw, 0, 0, self._gw, self._gh)
        self._grid_pil = grid_pil

    def _update_overlay(self, width_mm):
        import io as _io2
        arr = next((self._orig_arrays.get(k) for k in ("valley_map", "binary")
                    if self._orig_arrays.get(k) is not None), None)
        if not isinstance(arr, np.ndarray):
            return
        sw = max(1, int(width_mm * self._px_mm))
        ih, iw_arr = arr.shape[:2]
        sh = max(1, int(sw * ih / iw_arr))
        finger_pil = self._PilImage.fromarray(normalize_gray(arr), "L").convert("RGBA")
        finger_pil = finger_pil.resize((sw, sh), self._PilImage.LANCZOS)
        combined = self._grid_pil.copy().convert("RGBA")
        combined.paste(finger_pil, (0, 0), finger_pil)
        buf = _io2.BytesIO(); combined.convert("RGB").save(buf, format="PNG")
        iw = widgets.Image(value=buf.getvalue(), format="png")
        layer = self._mesh_canvas[1]
        with self._hc(layer):
            layer.clear()
            layer.draw_image(iw, 0, 0, self._gw, self._gh)

    def _draw_bg(self):
        self._draw_grid()
        self._apply(self._slider.value)

    def _on_slider(self, change):
        if hasattr(self, "_grid_pil"):
            self._apply(change['new'])

    def _apply(self, width_mm):
        self._update_overlay(width_mm)
        if self._img is None:
            return
        dpi = self._dpi_value()
        orig_h, orig_w = self._img.shape[:2]
        target_w_px = width_mm / 25.4 * dpi
        self._scale_factor = target_w_px / orig_w
        for key in self._RESCALE_KEYS:
            arr = self._orig_arrays.get(key)
            if not isinstance(arr, np.ndarray):
                continue
            new_w = max(1, round(arr.shape[1] * self._scale_factor))
            new_h = max(1, round(arr.shape[0] * self._scale_factor))
            interp = cv2.INTER_AREA if self._scale_factor < 1 else cv2.INTER_LANCZOS4
            self.context[key] = cv2.resize(arr, (new_w, new_h), interpolation=interp)
        phys_h = orig_h / orig_w * width_mm
        out_w = max(1, round(orig_w * self._scale_factor))
        out_h = max(1, round(orig_h * self._scale_factor))
        self._size_result.value = (
            f"<span style='color:#28a745;font-weight:bold'>"
            f"{width_mm:.1f}×{phys_h:.1f} mm</span>"
            f"&nbsp; <small style='color:#555'>{out_w}×{out_h} px @ {dpi:.0f} DPI</small>")

    def get_scale_factor(self):
        return self._scale_factor


# ── Screen preset selector ─────────────────────────────────────────────────
_SCREEN_PRESETS = {
    '\U0001f5a5️ Monitor  —  24" FHD  (Acer B247W, faculty PCs)': 1 / 0.2745,
    '\U0001f5a5️ Monitor  —  27" QHD  (~109 PPI)':                 109 / 25.4,
    '\U0001f5a5️ Monitor  —  27" 4K   (~163 PPI)':                 163 / 25.4,
    '\U0001f4bb Laptop   —  ThinkPad 14" FHD  (~157 PPI)':              157 / 25.4,
    '\U0001f4bb Laptop   —  ThinkPad 15.6" FHD (~141 PPI)':             141 / 25.4,
    '\U0001f4bb Laptop   —  ThinkPad 14" 2K  (~210 PPI)':               210 / 25.4,
    '\U0001f4bb Laptop   —  ThinkPad 14" 4K  (~315 PPI)':               315 / 25.4,
    '\U0001f34f Laptop   —  MacBook Air 13"  (~224 PPI)':               224 / 25.4,
    '\U0001f34f Laptop   —  MacBook Pro 14"  (~254 PPI)':               254 / 25.4,
}

_screen_sel = widgets.Dropdown(
    options=list(_SCREEN_PRESETS.keys()) + ['Custom pixel pitch'],
    value='\U0001f5a5️ Monitor  —  24" FHD  (Acer B247W, faculty PCs)',
    description='Monitor:',
    layout=widgets.Layout(width='560px'),
    style={'description_width': '70px'},
)
_custom_pitch = widgets.BoundedFloatText(
    value=0.2745, min=0.05, max=2.0, step=0.0001,
    description='Pitch (mm):',
    layout=widgets.Layout(width='210px', display='none'),
    style={'description_width': '80px'},
)
_px_mm_lbl = widgets.HTML('')

scale_widget = None
_cal_snapshots: dict = {}
_step4_out = widgets.Output()


def _launch_scale_widget():
    global scale_widget, _cal_snapshots
    for key in ScaleCalibrationWidget._RESCALE_KEYS:
        if key not in _cal_snapshots:
            arr = pipeline.context.get(key)
            if isinstance(arr, np.ndarray):
                _cal_snapshots[key] = arr.copy()
        else:
            pipeline.context[key] = _cal_snapshots[key].copy()

    ScaleCalibrationWidget._SCREEN_PX_MM = SCREEN_PX_PER_MM
    sw = ScaleCalibrationWidget(pipeline.context)

    unified = widgets.VBox([
        widgets.HTML(
            "<b style='font-size:1.05em'>Step 3 — Scale calibration</b>"
            "<hr style='margin:6px 0 10px 0'>"),
        widgets.HTML("<b>3a — Select your monitor</b>"),
        widgets.HBox(
            [_screen_sel, _custom_pitch, _px_mm_lbl],
            layout=widgets.Layout(align_items='center', margin='4px 0 0 0')),
        widgets.HTML("<hr style='margin:10px 0'>"),
        widgets.HTML(
            "<b>3b — Fingerprint size</b><br>"
            "<span style='font-size:0.95em'>Hold your real finger against the screen. "
            "Drag <b>Width</b> until the overlay matches — size is applied automatically.</span>"),
        widgets.HTML("<div style='margin:6px 0'></div>"),
        sw.inner,
    ], layout=widgets.Layout(
        border="1px solid #ccc", padding="14px",
        width=f"{sw._gw + 420}px"))

    _step4_out.clear_output(wait=True)
    with _step4_out:
        display(unified)
    sw._draw_bg()
    globals()['scale_widget'] = sw


def _on_screen_change(_=None):
    global SCREEN_PX_PER_MM
    if _screen_sel.value == 'Custom pixel pitch':
        _custom_pitch.layout.display = ''
        px_mm = 1.0 / _custom_pitch.value
    else:
        _custom_pitch.layout.display = 'none'
        px_mm = _SCREEN_PRESETS[_screen_sel.value]
    SCREEN_PX_PER_MM = px_mm
    _px_mm_lbl.value = (
        f"<span style='color:#555;font-size:0.9em'>"
        f"&nbsp; px/mm: <b>{px_mm:.4f}</b></span>")
    _launch_scale_widget()


_screen_sel.observe(_on_screen_change, names='value')
_custom_pitch.observe(
    lambda _: _screen_sel.value == 'Custom pixel pitch' and _on_screen_change(),
    names='value')

display(_step4_out)
_on_screen_change()

Output()

In [5]:
from PIL import Image as _PilSave
import ipywidgets as _w
from IPython.display import display as _d

try:    _no_image = IMAGE_PATH is None
except: _no_image = True
try:    _cal_ok = scale_widget.get_scale_factor() is not None
except: _cal_ok = False
try:    _ridge = pipeline.context.get('ridge_map')
except: _ridge = None

if _no_image:
    print('No image loaded — run Step 1 first.')
elif not _cal_ok:
    print('Run Step 3 first: select monitor and set width.')
elif _ridge is None:
    print('Run Step 2 first to generate the ridge map.')
else:
    import numpy as _np
    _arr = _np.where(_ridge > 0, 0, 255).astype(_np.uint8)   # ridges = black, background = white
    out_path = IMAGE_PATH.parent / f'{IMAGE_PATH.stem}_ridges_black.png'
    _dpi     = PRINTER_DPI
    _h, _w_px = _arr.shape[:2]
    _w_mm, _h_mm = _w_px / _dpi * 25.4, _h / _dpi * 25.4
    _PilSave.fromarray(_arr).save(str(out_path), dpi=(_dpi, _dpi))
    print(f'Saved → {out_path}')
    print(f'Print size: {_w_mm:.1f} × {_h_mm:.1f} mm  ({_w_px} × {_h} px @ {_dpi:.0f} DPI)')
    if not (10 <= min(_w_mm,_h_mm) <= 30 and 15 <= max(_w_mm,_h_mm) <= 50):
        print(f'⚠ Size looks wrong — re-check monitor selection in Step 3.')
    else:
        _d(_w.HTML("<div style='padding:8px;background:#eaf4ea;border-left:4px solid #28a745'>"
                   "✓ Saved! Print the PNG in Step 4.</div>"))

Saved → /home/x232886/PycharmProjects/poznejfi/data/finger_ridges_black.png
Print size: 27.5 × 43.5 mm  (650 × 1027 px @ 600 DPI)


HTML(value="<div style='padding:8px;background:#eaf4ea;border-left:4px solid #28a745'>✓ Saved! Print the PNG i…

---
## 4. DIY Guide 🛠️
1. **Print** `data/<name>_ridges_black.png` on **transparent foil**.
2. **Cover** the print with a thin layer of **Wood Glue (Lepidlo)**.
3. **Dry** completely (a few hours).
4. **Peel** it off — test it on your phone!